# Refraction-aware speed-of-sound autofocusing with bent rays

In the [differentiable beamforming example](./dbua_example.ipynb) we estimated a speed-of-sound (SoS) map by minimizing the common midpoint phase error (CMPE) with gradient descent. The beamforming delays in that example are integrated along *straight* rays between each element and each pixel. In a heterogeneous medium, sound refracts at every change in sound speed, so the true propagation paths are bent and the straight-ray delays are biased exactly where the SoS map is interesting.

In this notebook we use **UltraBend**, a refraction-aware propagation model that replaces the straight rays by bent rays traced through the current SoS map. Rays are marched through the slowness field as characteristics of the eikonal equation, their arrival times are deposited onto a regular travel-time grid, and the gradient with respect to the SoS map is computed with a replay adjoint that retraces the rays instead of storing them. In `zea` this is available as:

- [zea.beamform.bent_ray.compute_bent_ray_travel_times](../../_autosummary/zea.beamform.bent_ray.rst), the differentiable bent-ray travel-time model, and
- [zea.ops.BentRayTravelTimes](../../_autosummary/zea.ops.rst#zea.ops.BentRayTravelTimes), the pipeline operation that computes a travel-time map, which [zea.ops.TOFCorrection](../../_autosummary/zea.ops.rst#zea.ops.TOFCorrection) then uses for the delays.

For more information, see the UltraBend paper:
- Duelmer, F., Beuret, S., Frey, B.N., Azampour, M.F., Navab, N., Dahl, J.J. (2026). UltraBend: Refraction-Aware Propagation Modeling for Speed of Sound Imaging in Ultrasound. [OpenReview](https://openreview.net/forum?id=CIJzoHelIa)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tue-bmd/zea/blob/main/docs/source/notebooks/pipeline/ultrabend_example.ipynb)
&nbsp;
[![View on GitHub](https://img.shields.io/badge/GitHub-View%20Source-blue?logo=github)](https://github.com/tue-bmd/zea/blob/main/docs/source/notebooks/pipeline/ultrabend_example.ipynb)
&nbsp;
[![Hugging Face dataset](https://img.shields.io/badge/Hugging%20Face-Dataset-yellow?logo=huggingface)](https://huggingface.co/datasets/zeahub/simulations/blob/main/circular_inclusion_simulation.hdf5)

‼️ **Important:** This notebook is optimized for **GPU/TPU**. Code execution on a **CPU** may be very slow.

If you are running in Colab, please enable a hardware accelerator via:

**Runtime → Change runtime type → Hardware accelerator → GPU/TPU** 🚀.

In [ ]:
%%capture
%pip install zea

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"
os.environ["ZEA_DISABLE_CACHE"] = "1"
os.environ["ZEA_LOG_LEVEL"] = "INFO"

In [ ]:
import time

import keras
import matplotlib.pyplot as plt
import numpy as np
from keras import ops
from keras.utils import Progbar

import zea
from zea.backend.autograd import AutoGrad
from zea.backend.optimizer import adam
from zea.beamform.bent_ray import compute_bent_ray_travel_times, straight_ray_travel_times
from zea.ops import (
    BentRayTravelTimes,
    Cast,
    CommonMidpointPhaseError,
    DelayAndSum,
    EnvelopeDetect,
    LogCompress,
    Normalize,
    PatchedGrid,
    ReshapeGrid,
    TOFCorrection,
)
from zea.visualize import set_mpl_style

We will work with the GPU if available, and initialize using `init_device` to pick the best available device. Also, (optionally), we will set the matplotlib style for plotting.

In [ ]:
zea.init_device(verbose=False)
set_mpl_style()

The parameters below control the optimization. `n_rays` is the number of bent rays launched per element.

In [ ]:
num_iterations = 200
step_size = 1
n_rays = 256
file_path = "hf://zeahub/simulations/circular_inclusion_simulation.hdf5"

## Bent rays versus straight rays

Before estimating anything, let's look at what the propagation model changes on a medium we know. We create a SoS map with a circular inclusion that is 30 m/s faster than the background, similar to the inclusion phantom in the paper, and compute the one-way travel times from the center element of a linear array to every point of a travel-time grid, once with straight rays and once with bent rays.

In [ ]:
phantom_x = np.linspace(-15e-3, 15e-3, 61, dtype="float32")
phantom_z = np.linspace(0, 40e-3, 81, dtype="float32")
pz, px = np.meshgrid(phantom_z, phantom_x, indexing="ij")

# Circular inclusion with a smooth (2 mm) edge
radius = np.hypot(px, pz - 15e-3)
phantom_sos = 1540 + 30 / (1 + np.exp((radius - 5e-3) / 0.5e-3))
phantom_sos = phantom_sos.astype("float32")

# A single element at the center of the array
element = np.zeros((1, 3), dtype="float32")

tt_grid_x = np.linspace(-15e-3, 15e-3, 241, dtype="float32")
tt_grid_z = np.linspace(0, 40e-3, 321, dtype="float32")

bent = compute_bent_ray_travel_times(
    phantom_sos, phantom_x, phantom_z, element, tt_grid_x, tt_grid_z, n_rays=1024
)
gz, gx = np.meshgrid(tt_grid_z, tt_grid_x, indexing="ij")
nodes = np.stack([gx.ravel(), np.zeros(gx.size), gz.ravel()], axis=-1).astype("float32")
straight = straight_ray_travel_times(phantom_sos, phantom_x, phantom_z, element, nodes)
straight = ops.reshape(straight, bent.shape)

difference_ns = ops.convert_to_numpy(bent - straight)[0] * 1e9

The difference between both models is zero above the inclusion, where the rays have not met any change in sound speed. Below the inclusion, straight rays overestimate the travel time through its edges. Bent rays pass through the faster inclusion along shorter-time paths: by Fermat's principle the first arrival never takes longer than the straight path, which is why the difference is negative.

In [ ]:
extent = [tt_grid_x[0] * 1e3, tt_grid_x[-1] * 1e3, tt_grid_z[-1] * 1e3, tt_grid_z[0] * 1e3]
fig, (ax_sos, ax_diff) = plt.subplots(1, 2, figsize=(10, 4.5))

im = ax_sos.imshow(phantom_sos, extent=extent, cmap="jet")
ax_sos.plot(0, 0, "wv", markersize=8)
ax_sos.set_title("Speed of sound [m/s]")
fig.colorbar(im, ax=ax_sos, fraction=0.05, pad=0.02)

limit = np.abs(difference_ns).max()
im = ax_diff.imshow(difference_ns, extent=extent, cmap="RdBu_r", vmin=-limit, vmax=limit)
ax_diff.set_title("Bent minus straight travel time [ns]")
fig.colorbar(im, ax=ax_diff, fraction=0.05, pad=0.02)

for ax in (ax_sos, ax_diff):
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("z [mm]")
plt.tight_layout()
plt.show()

## Load dataset
Now let's estimate a SoS map from data. We use the same dataset as in the differentiable beamforming example: a circular inclusion in an isoechoic medium, simulated in [k-Wave](http://www.k-wave.org/) and stored in the [zea data format](../../data-acquisition.rst). It will automatically load the dataset from our [Hugging Face](https://huggingface.co/zeahub) repository.

In [ ]:
with zea.File(file_path) as file:
    data_frame = file.data.raw_data[:]
    parameters = file.load_parameters()

## Setting up the pipelines

We define the field of view, the f-number and a coarse grid of points where the CMPE loss is evaluated. As in the differentiable beamforming example, these points are resampled every iteration.

In [ ]:
xlims = (-15e-3, 15e-3)
zlims = (0e-3, 30e-3)
f_number = 0.7

grid_size_x = grid_size_z = 25

parameters.update(
    xlims=xlims,
    zlims=zlims,
    grid_size_x=grid_size_x,
    grid_size_z=grid_size_z,
    f_number=f_number,
)

We build two loss pipelines that only differ in the propagation model:

- **StraightRay**: [zea.ops.TOFCorrection](../../_autosummary/zea.ops.rst#zea.ops.TOFCorrection) receives the `sos_map` and integrates the slowness along straight rays.
- **UltraBend**: [zea.ops.BentRayTravelTimes](../../_autosummary/zea.ops.rst#zea.ops.BentRayTravelTimes) first traces bent rays through the `sos_map` and produces a `travel_time_map`, which `TOFCorrection` interpolates at every pixel. Note that the ray tracing happens *before* the `PatchedGrid`, such that the rays are traced once per SoS map and not once per patch of pixels.

In [ ]:
def make_loss_pipeline(propagation):
    patched = PatchedGrid(
        [TOFCorrection(), CommonMidpointPhaseError()],
        num_patches=grid_size_z * grid_size_x,
    )
    operations = [Cast(dtype="float32"), patched, ReshapeGrid()]
    if propagation == "bent":
        operations.insert(1, BentRayTravelTimes(n_rays=n_rays))
    return zea.Pipeline(operations, jit_options="pipeline")


loss_pipelines = {
    "StraightRay": make_loss_pipeline("straight"),
    "UltraBend": make_loss_pipeline("bent"),
}
inputs = loss_pipelines["StraightRay"].prepare_parameters(parameters)

for name, pipeline in loss_pipelines.items():
    print(f"{name}: {pipeline}")

For visualization we also set up a B-mode pipeline on a denser grid. It also uses the bent-ray travel times, such that the B-mode is beamformed with the same propagation model that is being optimized.

In [ ]:
width, height = xlims[1] - xlims[0], zlims[1] - zlims[0]
wavelength = parameters.sound_speed / parameters.center_frequency

image_grid_size_x = int(width / (0.5 * wavelength) / 4) + 1
image_grid_size_z = int(height / (0.5 * wavelength) / 4) + 1

parameters_plot = parameters.copy()
parameters_plot.update(grid_size_x=image_grid_size_x, grid_size_z=image_grid_size_z)


def make_image_pipeline(propagation):
    patched = PatchedGrid(
        [TOFCorrection(), DelayAndSum()],
        num_patches=image_grid_size_x * image_grid_size_z,
    )
    operations = [Cast(dtype="float32"), patched, ReshapeGrid()]
    if propagation == "bent":
        operations.insert(1, BentRayTravelTimes(n_rays=n_rays))
    operations += [EnvelopeDetect(), Normalize(), LogCompress()]
    return zea.Pipeline(operations, jit_options="pipeline")


image_pipelines = {
    "StraightRay": make_image_pipeline("straight"),
    "UltraBend": make_image_pipeline("bent"),
}
inputs_plot = image_pipelines["StraightRay"].prepare_parameters(parameters_plot)

## Speed of sound grid and initialization

The SoS map is defined on a coarse grid covering the field of view. Following the paper, we initialize it with the homogeneous sound speed that has the lowest CMPE. In a homogeneous medium bent rays are straight, so this survey uses the StraightRay pipeline.

In [ ]:
sos_grid_x = ops.linspace(xlims[0], xlims[1], 40)
sos_grid_z = ops.linspace(zlims[0], zlims[1], 40)
sos_shape = (sos_grid_z.shape[0], sos_grid_x.shape[0])


def cmpe_map(pipeline, sos_map, pipeline_inputs, flatgrid=None):
    if flatgrid is not None:
        pipeline_inputs = {**pipeline_inputs, "flatgrid": flatgrid}
    out = pipeline(
        data=data_frame,
        sos_map=sos_map,
        sos_grid_x=sos_grid_x,
        sos_grid_z=sos_grid_z,
        **pipeline_inputs,
    )
    return out["data"]


survey_speeds = np.arange(1420, 1601, 10)
survey_loss = [
    float(ops.nanmean(cmpe_map(loss_pipelines["StraightRay"], c * ops.ones(sos_shape), inputs)))
    for c in survey_speeds
]
initial_sound_speed = float(survey_speeds[np.argmin(survey_loss)])

plt.figure(figsize=(5, 3))
plt.plot(survey_speeds, survey_loss, "o-")
plt.axvline(initial_sound_speed, color="gray", linestyle="--")
plt.xlabel("Homogeneous speed of sound [m/s]")
plt.ylabel("Mean CMPE [rad]")
plt.title(f"Initial speed of sound: {initial_sound_speed:.0f} m/s")
plt.show()

## Loss function and optimization loop

The loss combines the mean CMPE with a smoothness regularizer on the SoS map. The paper uses an anisotropic regularizer that penalizes lateral variations five times more than axial ones, because the limited angular coverage of a linear array constrains the lateral structure less. We use the quadratic smoothness term of the differentiable beamforming example with this 5:1 lateral-to-axial weighting.

In [ ]:
lateral_weight, axial_weight = 5.0, 1.0


def loss_fn(sos_map, loss_pipeline, inputs, flatgrid):
    dx_sos = sos_grid_x[1] - sos_grid_x[0]
    dz_sos = sos_grid_z[1] - sos_grid_z[0]
    metric = cmpe_map(loss_pipeline, sos_map, inputs, flatgrid)
    metric_loss = ops.mean(ops.nan_to_num(metric, nan=0.0))
    tvz = ops.mean(ops.square(ops.diff(sos_map, axis=0)))
    tvx = ops.mean(ops.square(ops.diff(sos_map, axis=1)))
    smoothness = (lateral_weight * tvx + axial_weight * tvz) / (lateral_weight + axial_weight)
    variation_loss = 2 * smoothness * 1e2 * dx_sos * dz_sos
    return metric_loss + variation_loss


def resample_grid(n_pix, seed):
    seed_generator = keras.random.SeedGenerator(seed)
    x = keras.random.uniform((n_pix,), xlims[0] + 5e-3, xlims[1], seed=seed_generator)
    z = keras.random.uniform((n_pix,), zlims[0], zlims[1], seed=seed_generator)
    return ops.stack([x, ops.zeros_like(x), z], axis=-1)

We run the same optimization, with the same sequence of resampled loss points, for both propagation models.

In [ ]:
def optimize(name):
    loss_pipeline = loss_pipelines[name]
    autograd = AutoGrad()
    autograd.set_function(loss_fn)

    init_fn, update_fn, _ = adam(step_size)
    sos_map = initial_sound_speed * ops.ones(sos_shape)
    opt_state = init_fn(sos_map)
    losses = []

    print(name)
    progbar = Progbar(num_iterations)
    start = time.perf_counter()
    for i in range(num_iterations):
        flatgrid = resample_grid(inputs["flatgrid"].shape[0], seed=i)
        grad, loss = autograd.gradient_and_value(
            sos_map, loss_pipeline=loss_pipeline, inputs=inputs, flatgrid=flatgrid
        )
        sos_map, *state = update_fn(grad, opt_state)
        opt_state = (sos_map, *state)
        losses.append(float(loss))
        progbar.update(i + 1, [("loss", float(loss))])
    seconds = (time.perf_counter() - start) / num_iterations
    print(f"{seconds:.2f} s per iteration")
    return ops.convert_to_numpy(sos_map), losses


results = {name: optimize(name) for name in loss_pipelines}

## Results

Finally, we compare the estimated SoS maps, the B-mode images beamformed with them, and the loss curves of both propagation models.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
sos_extent = [xlims[0] * 1e3, xlims[1] * 1e3, zlims[1] * 1e3, zlims[0] * 1e3]
all_maps = np.stack([sos for sos, _ in results.values()])
vmin, vmax = np.percentile(all_maps, [1, 99])

for row, (name, (sos_map, _)) in enumerate(results.items()):
    bmode = image_pipelines[name](
        data=data_frame,
        sos_map=sos_map,
        sos_grid_x=sos_grid_x,
        sos_grid_z=sos_grid_z,
        **inputs_plot,
    )["data"][0]

    ax_bmode, ax_sos = axes[row]
    ax_bmode.imshow(bmode, extent=sos_extent, cmap="gray", vmin=-60, vmax=0)
    ax_bmode.set_title(f"{name}: B-mode")
    im = ax_sos.imshow(sos_map, extent=sos_extent, cmap="jet", vmin=vmin, vmax=vmax)
    ax_sos.set_title(f"{name}: speed of sound [m/s]")
    fig.colorbar(im, ax=ax_sos, fraction=0.05, pad=0.02)

for ax in axes.ravel():
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("z [mm]")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5, 3))
for name, (_, losses) in results.items():
    plt.plot(losses, label=name)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.show()

Tracing bent rays is more expensive than integrating along straight rays, so each UltraBend iteration takes longer. The ray tracer's memory use does not grow with the number of ray steps, because the replay adjoint retraces the rays in the backward pass instead of storing them. The accuracy of the travel times can be traded off against run time with the arguments of [zea.ops.BentRayTravelTimes](../../_autosummary/zea.ops.rst#zea.ops.BentRayTravelTimes), most importantly `n_rays` and `grid_upsample`.